In [ ]:

# Clone the ReStock AI repository
!git clone -b nour/t09-baseline https://github.com/raghad612/sic-project-restockai.git


In [ ]:

# Mount Google Drive to access the shared ReStock AI files
from google.colab import drive

drive.mount('/content/drive')


In [ ]:

# Load the shared feature dataset using the project's configured path
import sys
import pandas as pd

sys.path.insert(0, "/content/sic-project-restockai")

from src.config import FEATURES

df = pd.read_parquet(FEATURES)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Columns:", df.columns.tolist())


In [ ]:

# Verify the T9 chronological fit, validation, and test splits
from src.split import split_by_date

fit, validation, test = split_by_date(df)

# Check the required date boundaries
assert fit["date"].max() <= pd.Timestamp("2017-03-05")
assert validation["date"].min() == pd.Timestamp("2017-03-06")
assert validation["date"].max() == pd.Timestamp("2017-04-30")
assert test["date"].min() == pd.Timestamp("2017-05-01")
assert test["date"].max() == pd.Timestamp("2017-07-31")

# Flagged rows must not enter the fit split
assert not fit["fill_after_last_train"].any()

# Ensure there is no date overlap between splits
assert fit["date"].max() < validation["date"].min()
assert validation["date"].max() < test["date"].min()

print("PASS: Chronological split boundaries are correct.")
print("PASS: No flagged rows in fit.")
print("PASS: No date overlap between splits.")


In [ ]:

# Verify the exclusion rules for validation and test
for name, data, expected in [
    ("Validation", validation, 142411),
    ("Test", test, 232327),
]:
    missing_lag = data["lag_7"].isna()
    flagged = data["fill_after_last_train"]

    excluded = missing_lag | flagged
    eligible = data.loc[~excluded]

    print(f"\n{name}")
    print("Total rows:", len(data))
    print("Missing lag_7:", missing_lag.sum())
    print("Flagged rows:", flagged.sum())
    print("Overlapping exclusions:", (missing_lag & flagged).sum())
    print("Unique excluded rows:", excluded.sum())
    print("Eligible scoring rows:", len(eligible))

    assert len(eligible) == expected
    assert eligible["lag_7"].notna().all()
    assert not eligible["fill_after_last_train"].any()

    print("PASS: Exclusion rules are correct.")


In [ ]:

# Verify the saved baseline predictions against the existing lag_7
from src.config import BASELINE_PREDICTIONS

saved = pd.read_parquet(BASELINE_PREDICTIONS)

expected = pd.concat([
    validation.loc[
        validation["lag_7"].notna() &
        ~validation["fill_after_last_train"]
    ],
    test.loc[
        test["lag_7"].notna() &
        ~test["fill_after_last_train"]
    ]
], ignore_index=True)

expected = expected[
    ["date", "store_nbr", "item_nbr", "lag_7", "target"]
].rename(columns={"lag_7": "prediction"})

# Check schema, row counts, and actual prediction values
assert saved.columns.tolist() == [
    "date", "store_nbr", "item_nbr", "prediction", "target"
]
assert len(saved) == 374738

pd.testing.assert_frame_equal(
    saved.reset_index(drop=True),
    expected.reset_index(drop=True)
)

assert saved.isna().sum().sum() == 0

print("PASS: Prediction columns are correct.")
print("PASS: All 374,738 prediction rows match the eligible data.")
print("PASS: Every prediction equals the existing lag_7 value.")
print("PASS: No missing values in saved predictions.")


In [ ]:

# Verify the saved baseline metrics
import numpy as np
from src.config import BASELINE_METRICS
from src.metrics import mae, wape

metrics_df = pd.read_csv(BASELINE_METRICS)

assert metrics_df.columns.tolist() == [
    "period", "family", "unit_type",
    "rows", "MAE", "WAPE_percent"
]
assert len(metrics_df) == 31

KG_FAMILIES = {"POULTRY", "MEATS", "PRODUCE"}
MIXED_FAMILIES = {"PREPARED FOODS", "FROZEN FOODS"}

for name, data, expected_mae, expected_wape in [
    ("validation", validation, 14.512524369350741, 59.237007349962376),
    ("test", test, 14.45207598240611, 59.08792347001414),
]:
    eligible = data.loc[
        data["lag_7"].notna() &
        ~data["fill_after_last_train"]
    ]

    count_rows = eligible.loc[
        ~eligible["family"].astype(str).str.strip().str.upper().isin(
            KG_FAMILIES | MIXED_FAMILIES
        )
    ]

    calculated_mae = mae(count_rows["target"], count_rows["lag_7"])
    calculated_wape = wape(count_rows["target"], count_rows["lag_7"])

    saved_row = metrics_df.loc[
        (metrics_df["period"] == name) &
        (metrics_df["family"] == "COUNT_OVERALL")
    ]

    assert len(saved_row) == 1
    assert saved_row.iloc[0]["unit_type"] == "count"
    assert saved_row.iloc[0]["rows"] == len(count_rows)

    assert np.isclose(calculated_mae, expected_mae)
    assert np.isclose(calculated_wape, expected_wape)
    assert np.isclose(saved_row.iloc[0]["MAE"], calculated_mae)
    assert np.isclose(saved_row.iloc[0]["WAPE_percent"], calculated_wape)

    print(f"\n{name.capitalize()}")
    print("Count-family rows:", len(count_rows))
    print(f"MAE: {calculated_mae:.6f}")
    print(f"WAPE: {calculated_wape:.6f}%")
    print("PASS: Saved overall metrics match recalculated metrics.")

print("\nPASS: Metrics CSV structure and official overall results verified.")


In [ ]:

# Verify every saved per-family metric against the eligible data
for name, data in [("validation", validation), ("test", test)]:
    eligible = data.loc[
        data["lag_7"].notna() &
        ~data["fill_after_last_train"]
    ]

    saved_families = metrics_df.loc[
        (metrics_df["period"] == name) &
        (metrics_df["family"] != "COUNT_OVERALL")
    ]

    assert len(saved_families) == eligible["family"].nunique()

    for family, group in eligible.groupby("family", observed=True):
        family_name = str(family).strip().upper()

        if family_name in KG_FAMILIES:
            expected_type = "kg"
        elif family_name in MIXED_FAMILIES:
            expected_type = "mixed"
        else:
            expected_type = "count"

        saved_row = saved_families.loc[
            saved_families["family"] == family_name
        ]

        assert len(saved_row) == 1
        row = saved_row.iloc[0]

        assert row["unit_type"] == expected_type
        assert row["rows"] == len(group)
        assert np.isclose(
            row["MAE"],
            mae(group["target"], group["lag_7"])
        )
        assert np.isclose(
            row["WAPE_percent"],
            wape(group["target"], group["lag_7"]),
            equal_nan=True
        )

    print(f"{name.capitalize()}: {len(saved_families)} families verified.")
    print("PASS: Family classifications, row counts, MAE and WAPE.")

print("\nPASS: All saved per-family metrics are correct.")
